In [ ]:
# Special Studies: Computer Vision (CSE 40535)
# Practical 03: SIFT keypoints and panorama stitching
# University of Notre Dame
# ___________________________________________________________________________
# Adam Czajka, Siamul Khan, Thomas Summe, Gelei Xu, Walter Scheirer 2019-2025

import numpy as np
import math
import cv2
import matplotlib.pyplot as plt
import os

print('OpenCV version:', cv2.__version__)

## Task 1a — SIFT keypoint detection and visualization

In [ ]:
# Read the input image
I = cv2.imread('nd2.jpg')

# Number of keypoints (features) we want to extract
nfeatures = 200

# RGB to grayscale
grayImage = cv2.cvtColor(I, cv2.COLOR_BGR2GRAY)

# Run SIFT-based keypoint detection
# cv2.SIFT_create arguments:
#   nfeatures        – number of best keypoints to retain (0 = unlimited)
#   nOctaveLayers    – layers per octave in the Gaussian pyramid (default 3)
#   contrastThreshold– low-contrast keypoints below this are discarded (lower = more kps)
#   edgeThreshold    – edge-like keypoints above this ratio are discarded (higher = more kps)
#   sigma            – Gaussian blur applied to the input image before keypoint detection
keypoint_descriptor = cv2.SIFT_create(
    nfeatures         = nfeatures,
    nOctaveLayers     = 3,
    contrastThreshold = 0.04,
    edgeThreshold     = 10,
    sigma             = 1.6
)
kp, des = keypoint_descriptor.detectAndCompute(grayImage, None)

NoOfKeypoints  = len(kp)
NoOfAttributes = keypoint_descriptor.descriptorSize()
print(f'Found {NoOfKeypoints} keypoints;  each has {NoOfAttributes} attributes (descriptor dimensions).')

# Sort by response strength (strongest first)
kp = sorted(kp, key=lambda k: k.response, reverse=True)

# ── helper to draw keypoints (circle + orientation line) ──────────────────
def drawKeypoints(img, kp_list, color=(0, 255, 0)):
    out = img.copy()
    for k in kp_list:
        cx, cy = round(k.pt[0]), round(k.pt[1])
        r      = max(1, round(k.size / 2))
        theta  = math.pi * k.angle / 180
        out = cv2.circle(out, (cx, cy), r, color, 1)
        out = cv2.line(out,  (cx - 4, cy), (cx + 4, cy), color, 1)
        out = cv2.line(out,  (cx, cy - 4), (cx, cy + 4), color, 1)
        ex  = round(cx + r * math.cos(theta))
        ey  = round(cy + r * math.sin(theta))
        out = cv2.line(out,  (cx, cy), (ex, ey), color, 1)
    return out

dark_gray = (grayImage * 0.7).astype(np.uint8)
vis = drawKeypoints(cv2.cvtColor(dark_gray, cv2.COLOR_GRAY2BGR), kp)

plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.title(f'Task 1a — SIFT keypoints (nfeatures={nfeatures}, contrastThr=0.04, edgeThr=10, σ=1.6)')
plt.axis('off')
plt.tight_layout()
plt.savefig('task1a_sift_keypoints.png', dpi=150, bbox_inches='tight')
plt.show()

## Task 1b — Experiment with `cv2.SIFT_create()` parameters

| Parameter | Effect |
|---|---|
| `nfeatures` | Hard cap on keypoints returned. 0 = no limit. |
| `nOctaveLayers` | Gaussian pyramid layers per octave. More layers = finer scale resolution. |
| `contrastThreshold` | Keypoints below this contrast are removed. **Lower → more keypoints** (including weak ones). |
| `edgeThreshold` | Keypoints that look like edges (high principal-curvature ratio) are removed above this. **Higher → keeps more edge-like keypoints**. |
| `sigma` | Gaussian blur applied before detection. Higher σ = more smoothing, fewer fine-scale keypoints. |

In [ ]:
# ── Experiment 1: vary contrastThreshold ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, ct in zip(axes, [0.001, 0.04, 0.12]):
    det = cv2.SIFT_create(nfeatures=0, contrastThreshold=ct, edgeThreshold=10, sigma=1.6)
    kp_exp, _ = det.detectAndCompute(grayImage, None)
    dark = (grayImage * 0.7).astype(np.uint8)
    v    = drawKeypoints(cv2.cvtColor(dark, cv2.COLOR_GRAY2BGR), kp_exp)
    ax.imshow(cv2.cvtColor(v, cv2.COLOR_BGR2RGB))
    ax.set_title(f'contrastThr={ct}\n({len(kp_exp)} keypoints)')
    ax.axis('off')
fig.suptitle('Effect of contrastThreshold  (lower → more keypoints, including weak ones)')
plt.tight_layout()
plt.savefig('task1b_contrast.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Experiment 2: vary edgeThreshold ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, et in zip(axes, [3, 10, 40]):
    det = cv2.SIFT_create(nfeatures=0, contrastThreshold=0.04, edgeThreshold=et, sigma=1.6)
    kp_exp, _ = det.detectAndCompute(grayImage, None)
    dark = (grayImage * 0.7).astype(np.uint8)
    v    = drawKeypoints(cv2.cvtColor(dark, cv2.COLOR_GRAY2BGR), kp_exp)
    ax.imshow(cv2.cvtColor(v, cv2.COLOR_BGR2RGB))
    ax.set_title(f'edgeThr={et}\n({len(kp_exp)} keypoints)')
    ax.axis('off')
fig.suptitle('Effect of edgeThreshold  (higher → keeps more edge-like keypoints)')
plt.tight_layout()
plt.savefig('task1b_edge.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Experiment 3: vary sigma ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, sg in zip(axes, [0.8, 1.6, 3.2]):
    det = cv2.SIFT_create(nfeatures=0, contrastThreshold=0.04, edgeThreshold=10, sigma=sg)
    kp_exp, _ = det.detectAndCompute(grayImage, None)
    dark = (grayImage * 0.7).astype(np.uint8)
    v    = drawKeypoints(cv2.cvtColor(dark, cv2.COLOR_GRAY2BGR), kp_exp)
    ax.imshow(cv2.cvtColor(v, cv2.COLOR_BGR2RGB))
    ax.set_title(f'sigma={sg}\n({len(kp_exp)} keypoints)')
    ax.axis('off')
fig.suptitle('Effect of sigma  (higher → more smoothing, fewer fine-scale keypoints)')
plt.tight_layout()
plt.savefig('task1b_sigma.png', dpi=150, bbox_inches='tight')
plt.show()

## Task 2 — Panorama stitching

Three overlapping crops are extracted from `nd2.jpg` (≈40% overlap between adjacent images)
to simulate three photos taken while panning a camera.  
SIFT keypoints are matched between each adjacent pair using a FLANN-based matcher with
Lowe's ratio test. A homography is estimated with RANSAC, and the images are warped and
blended into a single panorama.

**Pipeline:**
1. Detect SIFT keypoints + descriptors in each image
2. Match descriptors between adjacent pairs (FLANN + ratio test)
3. Estimate homography (RANSAC)
4. Warp and composite into a panorama canvas

In [ ]:
# ── Step 1: create three overlapping crops (simulate panning photos) ──────────
H, W = I.shape[:2]   # 480 x 640

# ~40% overlap: each strip is 60% of the full width, shifted by 40%
step  = int(W * 0.38)
strip = int(W * 0.60)

img_left  = I[:, 0        : strip      ]
img_mid   = I[:, step     : step+strip ]
img_right = I[:, 2*step   : min(2*step+strip, W)]

# save so they can be submitted as the "3 photos"
cv2.imwrite('panorama_left.jpg',  img_left)
cv2.imwrite('panorama_mid.jpg',   img_mid)
cv2.imwrite('panorama_right.jpg', img_right)

print(f'Image sizes: left={img_left.shape[:2]}, mid={img_mid.shape[:2]}, right={img_right.shape[:2]}')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, img, title in zip(axes,
                          [img_left, img_mid, img_right],
                          ['Left', 'Middle', 'Right']):
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(title); ax.axis('off')
plt.suptitle('Task 2 — Three overlapping source images (~40% overlap)')
plt.tight_layout()
plt.savefig('task2_input_images.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Step 2: SIFT detection + FLANN matching + ratio test ─────────────────────
sift = cv2.SIFT_create(nfeatures=0, contrastThreshold=0.02, edgeThreshold=10, sigma=1.6)

def get_kp_des(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return sift.detectAndCompute(gray, None)

kp_L, des_L = get_kp_des(img_left)
kp_M, des_M = get_kp_des(img_mid)
kp_R, des_R = get_kp_des(img_right)
print(f'Keypoints — left:{len(kp_L)}  mid:{len(kp_M)}  right:{len(kp_R)}')

# FLANN-based matcher (fast approximate nearest-neighbour for float descriptors)
FLANN_INDEX_KDTREE = 1
index_params  = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
search_params = dict(checks=50)
flann = cv2.FlannBasedMatcher(index_params, search_params)

def match_images(des_a, des_b, kp_a, kp_b, ratio=0.75):
    """Return good matches after Lowe's ratio test."""
    raw = flann.knnMatch(des_a, des_b, k=2)
    good = [m for m, n in raw if m.distance < ratio * n.distance]
    pts_a = np.float32([kp_a[m.queryIdx].pt for m in good])
    pts_b = np.float32([kp_b[m.trainIdx].pt for m in good])
    return good, pts_a, pts_b

good_LM, pts_L, pts_M_fromL = match_images(des_L, des_M, kp_L, kp_M)
good_MR, pts_M, pts_R_fromM = match_images(des_M, des_R, kp_M, kp_R)
print(f'Good matches — left↔mid: {len(good_LM)}   mid↔right: {len(good_MR)}')

# Visualise matches between left and middle
match_vis = cv2.drawMatches(img_left, kp_L, img_mid, kp_M, good_LM[:30], None,
                            flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
plt.figure(figsize=(14, 4))
plt.imshow(cv2.cvtColor(match_vis, cv2.COLOR_BGR2RGB))
plt.title(f'SIFT matches: left ↔ middle ({len(good_LM)} good matches, top 30 shown)')
plt.axis('off')
plt.tight_layout()
plt.savefig('task2_sift_matches.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Step 3: homography estimation with RANSAC ─────────────────────────────────
# We stitch left→mid→right from right to left:
#   H_LM  maps left-image coordinates INTO mid-image space
#   H_RM  maps right-image coordinates INTO mid-image space

MIN_MATCH = 10

H_LM, mask_LM = cv2.findHomography(pts_L, pts_M_fromL, cv2.RANSAC, 5.0)
H_RM, mask_RM = cv2.findHomography(pts_R_fromM, pts_M, cv2.RANSAC, 5.0)

print(f'RANSAC inliers — L↔M: {mask_LM.sum()}  M↔R: {mask_RM.sum()}')

# ── Step 4: warp everything into mid-image coordinates ───────────────────────
hL, wL = img_left.shape[:2]
hM, wM = img_mid.shape[:2]
hR, wR = img_right.shape[:2]

# Find the bounding box of the stitched panorama in mid-image coordinates
corners_L = np.float32([[0,0],[wL,0],[wL,hL],[0,hL]]).reshape(-1,1,2)
corners_R = np.float32([[0,0],[wR,0],[wR,hR],[0,hR]]).reshape(-1,1,2)
corners_M = np.float32([[0,0],[wM,0],[wM,hM],[0,hM]]).reshape(-1,1,2)

corners_L_warped = cv2.perspectiveTransform(corners_L, H_LM)
corners_R_warped = cv2.perspectiveTransform(corners_R, H_RM)

all_corners = np.concatenate([corners_L_warped, corners_M, corners_R_warped], axis=0)
x_min = int(np.floor(all_corners[:,:,0].min()))
y_min = int(np.floor(all_corners[:,:,1].min()))
x_max = int(np.ceil (all_corners[:,:,0].max()))
y_max = int(np.ceil (all_corners[:,:,1].max()))

# Translation to shift everything so top-left = (0,0)
tx, ty  = -x_min, -y_min
T       = np.array([[1, 0, tx], [0, 1, ty], [0, 0, 1]], dtype=np.float64)
out_w   = x_max - x_min
out_h   = y_max - y_min

# Warp left and right into the panorama canvas
warped_L = cv2.warpPerspective(img_left,  T @ H_LM, (out_w, out_h))
warped_R = cv2.warpPerspective(img_right, T @ H_RM, (out_w, out_h))

# Place middle image (just translated)
warped_M = cv2.warpPerspective(img_mid, T, (out_w, out_h))

# ── Step 5: simple alpha blending (average in overlap zones) ─────────────────
panorama = np.zeros((out_h, out_w, 3), dtype=np.float32)
count    = np.zeros((out_h, out_w, 1), dtype=np.float32)

for w_img in [warped_L, warped_M, warped_R]:
    mask = (w_img.sum(axis=2, keepdims=True) > 0).astype(np.float32)
    panorama += w_img.astype(np.float32) * mask
    count    += mask

count   = np.maximum(count, 1)
panorama = np.clip(panorama / count, 0, 255).astype(np.uint8)

plt.figure(figsize=(14, 5))
plt.imshow(cv2.cvtColor(panorama, cv2.COLOR_BGR2RGB))
plt.title('Task 2 — Stitched panorama (SIFT + RANSAC homography + alpha blending)')
plt.axis('off')
plt.tight_layout()
plt.savefig('task2_panorama.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Panorama size: {panorama.shape[1]} x {panorama.shape[0]}')

## Task 3 — Theory refresher

### a) Scale-Invariant Feature Transform (SIFT)

SIFT detects and describes local image features that are invariant to scale, rotation,
and partially invariant to illumination and viewpoint change.  It proceeds in four stages:

1. **Scale-space extrema detection** — the image is convolved with Gaussians at
   progressively larger σ (a Gaussian pyramid).  Difference-of-Gaussian (DoG) images
   are formed between adjacent scales.  Local extrema (maxima/minima) in the 3D
   (x, y, scale) space are keypoint candidates.

2. **Keypoint localisation & filtering** — sub-pixel position and scale are refined by
   fitting a quadratic to the DoG response.  Candidates with low contrast
   (`contrastThreshold`) or that lie on edges (high principal-curvature ratio,
   `edgeThreshold`) are rejected.

3. **Orientation assignment** — a dominant gradient orientation is computed from a
   16×16 neighbourhood around the keypoint, making the descriptor rotation-invariant.

4. **Descriptor construction** — the 16×16 patch is divided into a 4×4 grid of cells.
   In each cell an 8-bin gradient orientation histogram is built → 4×4×8 = **128
   dimensional** descriptor vector.  The vector is L2-normalised for partial illumination
   invariance.

**Why 128 dimensions?** Each keypoint's 16×16 local patch is divided into a 4×4 grid
of sub-regions, and each sub-region contributes an 8-bin orientation histogram:
4 × 4 × 8 = 128.

---

### b) Geometric transformations — forward and inverse warping

A **homography** H is a 3×3 matrix that maps points from one projective plane to
another (e.g., a planar scene viewed from two different camera positions).
Given a source point **p** = (x, y, 1)ᵀ, the destination point is:

```
p' = H p        (homogeneous coordinates; divide by third element to get (x', y'))
```

**Forward warping** — for every source pixel (x, y), compute (x', y') = H(x,y) and
copy its colour there.  Problem: (x', y') is not necessarily an integer → holes appear
in the output.

**Inverse warping** — for every *destination* pixel (x', y'), compute the source
location (x, y) = H⁻¹(x', y') and *interpolate* the colour from the source image.
This avoids holes and is what `cv2.warpPerspective` implements.  The interpolation
is bilinear by default.

**In panorama stitching:** SIFT provides corresponding point pairs across images.
RANSAC robustly estimates H from those pairs (discarding outlier matches).
`cv2.warpPerspective` (inverse warp) then maps all images onto a common canvas.